# Actividad 15. Implementación de Técnicas de Programación Dinámica

**Materia:** Lenguajes y Computación

## Equipo

| Nombre completo | No. de cuenta |
|---|---|
| Jonathan Hernández Lazcano | 200417 |
| Camila Rodriguez Rosas | 194100 |

## Objetivo

1. Implementar la función `CUT_ROD` con **tabulación** (bottom-up).
2. Realizar un **benchmark** que compare el rendimiento de las 3 técnicas (recursiva, memoización y tabulación) variando el tamaño de la varilla.

## El Problema del corte de varillas (Rod Cutting Problem)

El problema consiste en que considerando una varilla de *n* pulgadas y una tabla de precios $p_n$ por pulgada, donde *i* = {1, 2, ..., n}, se pueda determinar el máximo ingreso por los cortes realizados o no, en la varilla.

Considerando el precio de cada corte en la varilla, es posible que no necesariamente se condisidere un corte.

### Tabla de precios por tamaño de corte

| Longitud (i) | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Precio | \$1 | \$5 | \$8 | \$9 | \$10 | \$17 | \$17 | \$20 | \$24 | \$30 |

Una variblla de *n* pulgadas se puede cortar de $2^{n-1}$ formas distintas, ya que cada pulgada *i* = {1, 2, ..., n-1} existen opciones para realizar los cortes.

### Solución recursiva (pura)

In [1]:
# importación de librerías
import time
import sys

In [2]:
# Acotamiento del tiempo de ejecución para los algoritmos recursivos
sys.setrecursionlimit(2000)

In [3]:
# Definición de la entrada constante
PRECIOS = (0, 1, 5, 8, 9, 10, 17, 17, 20, 24, 30)

In [4]:
# Implementación de la función recursiva pura
def cut_rod(p: list[int], n:int) -> int:
    """
    Función recursiva para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n.
    """
    if n == 0:
        return 0
    q = -1
    for i in range(1, n+1):
      # Evaluar todas las opciones de corte inicial de tamaño i e invocando de manera recursiva a la función para calcular el máximo ingreso con el remanente
      precio_i = p[i] if i < len(p) else 0
      q = max(q, precio_i + cut_rod(p, n-i))
    return q

### Implementación de las técnicas de programación dinámica

Como se pudo indentificar previamente, en el peor de los casos el problema del cálculo de la máxima ganancia en el corte de una varilla, esta dado por $O(2^{n})$.

Justamente la programación dinámica, busca evitar la redundancia en el cálculo de algunos cortes que previamente ya fueron evaluados a través de la implementación ya sea de una memoria caché (arreglo) o bien de una tabla.

#### Memorización (Top-Down)

Este técnica reutiliza la misma función recursiva pura y agrega una caché para verificar si el resultado de un cálculo ya fue evaluado previamente, en cuyo caso recupera el valor almacenado ($O(1)$).

In [5]:
# Definición de la función de memoización
def cut_rod_memoized(p: list[int], n:int) -> int:
    """
    Función recursiva para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n, implementando una función auxiliar de memoización
    """
    # Creación de un arreglo "r" de tamaño n+1 e inicilizado a -1 (valores desconocidos/no calculados)
    r = [-1] * (n+1)
    return cut_rod_aux(p, n, r)

def cut_rod_aux(p: list[int], n:int, r: list[int]) -> int:
    """
    Función auxiliar para la implementación de la técnica de memoización para verificar si el resultado de un cálculo ya fue evaluado previamente, en cuyo caso recupera el valor almacenado
    """
    # Verificación de resultado en memoria
    if r[n] >= 0:
      return r[n]

    # Caso de finalización/retorno
    if n==0:
      q=0
    else:
      q = -1
      for i in range(1, n+1):
        # Evaluar todas las opciones de corte inicial de tamaño i e invocando de manera recursiva a la función para calcular el máximo ingreso con el remanente
        precio_i = p[i] if i < len(p) else 0
        q = max(q, precio_i + cut_rod_aux(p, n-i, r))

    # Almacenamiento del resultado en memoria
    r[n] = q
    return q

#### Implementación de la técnica tabulación (Bottom-Up)

Esta técnica elimina por completo la recursión: resuelve primero los subproblemas más pequeños y los almacena en una tabla `r[0..n]`, de modo que al calcular `r[j]` todos los valores `r[j-i]` que necesita ya están disponibles.

$$r_0 = 0, \qquad r_j = \max_{1 \le i \le j} \left( p_i + r_{j-i} \right)$$

Como hay dos ciclos anidados, el tiempo es $O(n^2)$ y el espacio $O(n)$. Al no usar la pila de llamadas, tampoco depende del límite de recursión (`sys.setrecursionlimit`).

In [6]:
# Definición de la función de tabulación
def cut_rod_bottom_up(p: list[int], n:int) -> int:
    """
    Función iterativa para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n, llenando una tabla de abajo hacia arriba (bottom-up)
    """
    # Creación de la tabla "r" de tamaño n+1; r[0] = 0 porque una varilla de longitud 0 no genera ingreso
    r = [0] * (n+1)
    for j in range(1, n+1):
      q = -1
      for i in range(1, j+1):
        # Evaluar todas las opciones de corte inicial de tamaño i reutilizando el ingreso máximo ya calculado para el remanente j-i
        precio_i = p[i] if i < len(p) else 0
        q = max(q, precio_i + r[j-i])
      # Almacenamiento del resultado en la tabla
      r[j] = q
    return r[n]

Para conocer **cuáles son los cortes**, se extiende la tabulación guardando en un arreglo `s` el tamaño del primer corte óptimo para cada longitud `j`. Después se reconstruye la solución recorriendo `s` desde `n` hasta llegar a 0.

In [7]:
# Definición de la función de tabulación extendida (ingreso + cortes)
def extended_cut_rod_bottom_up(p: list[int], n:int) -> tuple[list[int], list[int]]:
    """
    Función iterativa que, además de la tabla de ingresos "r", construye la tabla "s" con el tamaño del primer corte óptimo para cada longitud
    """
    r = [0] * (n+1)
    s = [0] * (n+1)
    for j in range(1, n+1):
      q = -1
      for i in range(1, j+1):
        precio_i = p[i] if i < len(p) else 0
        if precio_i + r[j-i] > q:
          q = precio_i + r[j-i]
          # Registro del primer corte que produce el mejor ingreso hasta el momento
          s[j] = i
      r[j] = q
    return r, s

def obtener_cortes(s: list[int], n:int) -> list[int]:
    """
    Función para reconstruir la lista de cortes óptimos a partir de la tabla "s"
    """
    cortes = []
    while n > 0:
      cortes.append(s[n])
      n -= s[n]
    return cortes

In [8]:
# Verificación: las tres técnicas deben obtener el mismo ingreso máximo
for n in range(0, 21):
    esperado = cut_rod(PRECIOS, n)
    assert cut_rod_memoized(PRECIOS, n) == esperado, f"Memoización difiere en n={n}"
    assert cut_rod_bottom_up(PRECIOS, n) == esperado, f"Tabulación difiere en n={n}"
    r, s = extended_cut_rod_bottom_up(PRECIOS, n)
    assert r[n] == esperado and sum(PRECIOS[c] for c in obtener_cortes(s, n)) == esperado

print("Las tres técnicas coinciden para n = 0..20\n")
for n in (4, 7, 10):
    r, s = extended_cut_rod_bottom_up(PRECIOS, n)
    print(f"n = {n:2d} -> ingreso máximo = {r[n]:3d}, cortes = {obtener_cortes(s, n)}")

Las tres técnicas coinciden para n = 0..20

n =  4 -> ingreso máximo =  10, cortes = [2, 2]
n =  7 -> ingreso máximo =  18, cortes = [1, 6]
n = 10 -> ingreso máximo =  30, cortes = [10]


#### Benchmark de las 3 técnicas

Elabora una tabla que compare los tiempos para el cálculo de los ingresos máximos utilizando las tres técnicas, y adecuando las funciones para conocer cuáles serían los cortes.

Resultados obtenidos al ejecutar la libreta (Python 3.14, Apple Silicon); los tiempos pueden variar según el equipo:

| Longitud de la varilla | Recursivo | Memoización | Tabulación | Ingreso Máximo | Cortes |
| --- | --- | --- | --- | --- | --- |
| 5 | 2.75 µs | 1.25 µs | 0.91 µs | \$13 | [2, 3] |
| 10 | 78.92 µs | 3.94 µs | 2.91 µs | \$30 | [10] |
| 15 | 2.50 ms | 8.09 µs | 5.63 µs | \$43 | [2, 3, 10] |
| 20 | 80.81 ms | 14.74 µs | 9.96 µs | \$60 | [10, 10] |
| 25 | 2.71 s | 21.86 µs | 14.32 µs | \$73 | [2, 3, 10, 10] |
| 28 | 21.63 s | 27.48 µs | 18.20 µs | \$82 | [2, 6, 10, 10] |
| 30 | 86.69 s | 31.18 µs | 20.21 µs | \$90 | [10, 10, 10] |

Para cada longitud se mide el tiempo con `time.perf_counter`. Como la memoización y la tabulación tardan microsegundos, se promedian varias repeticiones; el recursivo puro se ejecuta una sola vez a partir de $n = 20$, ya que su costo es exponencial ($2^{n-1}$ formas de cortar: para $n = 30$ son más de 500 millones). **Nota:** esta celda tarda algunos minutos por el caso recursivo con $n = 28$ y $n = 30$.

In [9]:
from IPython.display import Markdown, display

def medir(func, p: list[int], n:int, repeticiones:int = 1) -> tuple[float, int]:
    """
    Función para medir el tiempo promedio (en segundos) de ejecutar func(p, n), regresando también su resultado
    """
    inicio = time.perf_counter()
    for _ in range(repeticiones):
      resultado = func(p, n)
    return (time.perf_counter() - inicio) / repeticiones, resultado

def formato_tiempo(segundos: float) -> str:
    """
    Función para expresar un tiempo en la unidad más legible (µs, ms o s)
    """
    if segundos < 1e-3:
      return f"{segundos * 1e6:.2f} µs"
    if segundos < 1:
      return f"{segundos * 1e3:.2f} ms"
    return f"{segundos:.2f} s"

TAMANOS = [5, 10, 15, 20, 25, 28, 30]
resultados = []

for n in TAMANOS:
    t_rec, q_rec = medir(cut_rod, PRECIOS, n, repeticiones=1 if n >= 20 else 10)
    t_mem, q_mem = medir(cut_rod_memoized, PRECIOS, n, repeticiones=1000)
    t_tab, q_tab = medir(cut_rod_bottom_up, PRECIOS, n, repeticiones=1000)
    assert q_rec == q_mem == q_tab, f"Las técnicas no coinciden en n={n}"
    r, s = extended_cut_rod_bottom_up(PRECIOS, n)
    resultados.append((n, t_rec, t_mem, t_tab, q_tab, obtener_cortes(s, n)))
    print(f"n = {n:2d} listo (recursivo: {formato_tiempo(t_rec)})")

tabla = "| Longitud de la varilla | Recursivo | Memoización | Tabulación | Recursivo / Tabulación | Ingreso Máximo | Cortes |\n"
tabla += "| --- | --- | --- | --- | --- | --- | --- |\n"
for n, t_rec, t_mem, t_tab, q, cortes in resultados:
    tabla += f"| {n} | {formato_tiempo(t_rec)} | {formato_tiempo(t_mem)} | {formato_tiempo(t_tab)} | {t_rec / t_tab:,.0f}x | \\${q} | {cortes} |\n"
display(Markdown(tabla))

n =  5 listo (recursivo: 2.75 µs)
n = 10 listo (recursivo: 78.92 µs)
n = 15 listo (recursivo: 2.50 ms)
n = 20 listo (recursivo: 80.81 ms)


n = 25 listo (recursivo: 2.71 s)


n = 28 listo (recursivo: 21.63 s)


n = 30 listo (recursivo: 86.69 s)


| Longitud de la varilla | Recursivo | Memoización | Tabulación | Recursivo / Tabulación | Ingreso Máximo | Cortes |
| --- | --- | --- | --- | --- | --- | --- |
| 5 | 2.75 µs | 1.25 µs | 0.91 µs | 3x | \$13 | [2, 3] |
| 10 | 78.92 µs | 3.94 µs | 2.91 µs | 27x | \$30 | [10] |
| 15 | 2.50 ms | 8.09 µs | 5.63 µs | 443x | \$43 | [2, 3, 10] |
| 20 | 80.81 ms | 14.74 µs | 9.96 µs | 8,117x | \$60 | [10, 10] |
| 25 | 2.71 s | 21.86 µs | 14.32 µs | 189,124x | \$73 | [2, 3, 10, 10] |
| 28 | 21.63 s | 27.48 µs | 18.20 µs | 1,188,413x | \$82 | [2, 6, 10, 10] |
| 30 | 86.69 s | 31.18 µs | 20.21 µs | 4,290,387x | \$90 | [10, 10, 10] |


#### Conclusiones

- **Recursivo puro:** su tiempo crece de forma exponencial, $O(2^n)$; aproximadamente se multiplica por 2 con cada pulgada adicional, porque recalcula los mismos subproblemas una y otra vez. Pasa de microsegundos en $n = 5$ a minutos en $n = 30$.
- **Memoización (top-down):** al guardar cada `r[n]` calculado, cada subproblema se resuelve una sola vez, por lo que el tiempo baja a $O(n^2)$. Aun así paga el costo de las llamadas recursivas y depende de la profundidad de la pila (`sys.setrecursionlimit`).
- **Tabulación (bottom-up):** también es $O(n^2)$ en tiempo y $O(n)$ en espacio, pero con ciclos simples; por eso suele ser la más rápida y no tiene límite de recursión.
- Las tres técnicas obtienen el **mismo ingreso máximo**; la versión extendida de la tabulación permite además recuperar los cortes óptimos (p. ej. para $n = 30$ conviene cortar tres piezas de 10 pulgadas).